In [29]:
from dotenv import load_dotenv
import pandas as pd
import os
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient
from qna_extractor import qna_extractor
import json

load_dotenv()


True

In [ ]:
from qna_extractor import qna_extractor

# Create extractor
extractor = qna_extractor(
    question="Thank you for resolving my issue!", 
    project="MIRA"
)

# Extract sentiments
level_1 = extractor.extract_sentiment_level_one()
print("Level 1 Sentiment:", level_1)
level_2 = extractor.extract_sentiment_level_two("Positive")
print("Level 2 Sentiment:", level_2)

# Results automatically stored in Azure AI Search index
#Positive-Grateful

2025-10-02 14:10:33 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
Level 1 Sentiment: 
{"sentiment_level_1": "Positive"}

2025-10-02 14:10:34 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
Level 2 Sentiment: 
{"sentiment_level_2": "Grateful"}



In [30]:
search_client = SearchClient("https://gpdacq-dev-1-eastus-ai-search-cog-svc.search.windows.net","transcripts-ucid-qna-v5",AzureKeyCredential(os.getenv("AZURE_SEARCH_API_KEY_EASTUS")))
results = search_client.search(select=['Ucid','question'],filter="last_modified_time gt 2025-10-06T17:05:00Z and last_modified_time lt 2025-10-06T17:05:03Z")

In [31]:
df = pd.DataFrame(results)
df.shape

(0, 0)

In [23]:
df.head(n=10)

""


In [ ]:
def extract_sentiments(row):
    extractor = qna_extractor(question=row['question'], project="MIRA")
    sentiment_level_1_json = extractor.extract_sentiment_level_one()
    cleaned_sentiment_1 = sentiment_level_1_json.strip()
    cleaned_sentiment_1 = cleaned_sentiment_1.replace("```json", "").replace("```", "").strip()
    cleaned_sentiment_1 = cleaned_sentiment_1.lstrip('\n').rstrip('\n')
    
    sentiment_level_1_data = json.loads(cleaned_sentiment_1)
    sentiment_level_1_value = sentiment_level_1_data.get('sentiment_level_1')
    #expanded_qna_1["sentiment_level_1"] = sentiment_level_1_value
    row['sentiment_level_1'] = sentiment_level_1_value 

    sentiment_level_2_json = extractor.extract_sentiment_level_two(sentiment_level_1_value)
    # Clean the response before parsing
    cleaned_sentiment_2 = sentiment_level_2_json.strip()
    cleaned_sentiment_2 = cleaned_sentiment_2.replace("```json", "").replace("```", "").strip()
    cleaned_sentiment_2 = cleaned_sentiment_2.lstrip('\n').rstrip('\n')
    
    sentiment_level_2_data = json.loads(cleaned_sentiment_2)
    #expanded_qna_1["sentiment_level_2"] = sentiment_level_2_data.get('sentiment_level_2')
    row['sentiment_level_2'] = sentiment_level_2_data.get('sentiment_level_2')
    return row 

df = df.apply(extract_sentiments, axis=1)

2025-10-06 14:39:20 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-06 14:39:20 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-06 14:39:21 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-06 14:39:22 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-06 14:39:22 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-06 14:39:23 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-06 14:39:23 - gpd_ai_companion_qna_batch - INFO - sentiment_level_1_extraction extraction completed successfully.
2025-10-06 14:39:24 - gpd_ai_companion_qna_batch - INFO - sentiment_level_2_extraction extraction completed successfully.
2025-10-06 14:39:25 - gp

In [19]:
df.head(n=10)

,question,Ucid,@search.score,@search.reranker_score,@search.highlights,@search.captions,sentiment_level_1,sentiment_level_2
0,Is the mobile app I have the correct one for m...,00099232791759763090,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquiry""}\n"
1,Do I need to do anything to remain enrolled in...,00066131731759763975,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquisitive""}\n"
2,Is the new plan going to be with United Health?,00070608701759765502,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquiry""}\n"
3,Are vision benefits included?,00070608701759765502,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquiry""}\n"
4,Which pharmacy do you normally use for your pr...,00067321741759765123,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquiry""}\n"
5,What should I do if I am not eligible for the ...,00098524121759764355,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquisitive""}\n"
6,Have there been any changes in my medications ...,00098524121759764355,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquisitive""}\n"
7,I used to have a PPO plan and I see you don’t ...,00099234841759763119,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquiry""}\n"
8,When does the Medicare Plan D sign-up period s...,00070044381759762514,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Informational Inquiry..."
9,Will I have to pay copays for medications unde...,00102005871759762368,1.0,None,None,None,"\n{""sentiment_level_1"": ""Neutral""}\n","\n{""sentiment_level_2"": ""Inquiry""}\n"


In [20]:
df.to_csv("transcripts_ucid_qna_with_sentiments.csv", index=False)

In [ ]:

extractor = qna_extractor(project="MIRA")
df['sentiment_level_1'] = df.apply(lambda row: extractor.extract_sentiment_level_one(row['question']), axis=1)
df['sentiment_level_2'] = df.apply(lambda row: extractor.extract_sentiment_level_two(row['sentiment_level_1'], row['question']), axis=1)

TypeError: qna_extractor.extract_sentiment_level_one() takes 1 positional argument but 2 were given

In [ ]:
df.head(n=10)